# 16 CBS 增强：基数冲突与 bypass


本课按百科条目写。朴素 CBS 见冲突就二分，树肥。增强（ICBS 等）少分支、仍最优。调度台在最后。

---

## 1. 定义

工业 CBS（ICBS、CBSH 等）在分裂前给冲突分类：

- **基数冲突（cardinal）：** 无论让 A 还是让 B，SOC **都上升**。应优先拆，否则树在无关冲突上浪费。
- **semi-cardinal：** 只有一方 SOC 上升。
- **non-cardinal：** 双方 SOC 都不升（有另一条同样长的路）。
- **bypass：** 某个孩子 SOC 不变且冲突数下降，用它 **替换** 当前节点，不把兄弟推进堆。
- **对称性**（走廊、矩形）：一次加一串约束，避免沿着走廊逐步等。

本仓库 `enhanced=True`：生成两个孩子后，若存在 SOC 不增的，只压那个（简化 bypass）。没有完整 cardinal MDDs。

---

## 2. 不变量

增强 **不放松最优性**：仍先弹出 SOC 最小可行叶。只减少扩展。bypass 的孩子 SOC=父 SOC，下界不变。

若 bypass 不检查冲突是否真下降，可能在两个等 SOC 节点间循环——库用「只压一个孩子、约束仍增加」避免原地踏步。

---

## 3. 伪代码（本课实现）

```
children ← []
for 每个分支约束:
    重搜该车，得 new_paths, new_soc
    children.append(...)
if 存在 new_soc == 当前 soc:
    只 push 那一个   # bypass
else:
    全部 push         # 双方都变贵 ≈ cardinal
```

完整 ICBS 还要用 MDD（该代价下的所有路径）判定 cardinal，并优先拆 cardinal。

---

## 4. 手算

父 SOC=10。孩子 A：SOC=10（另找一条等长路躲开冲突）。孩子 B：SOC=12。

bypass：只保留 A，不把 B 进堆。若 A 仍有别的冲突，下一轮再拆。最优性：B 的 12 不会比「沿 A 继续」的任何 ≤10 的解更好。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：bypass 决策（先看数字，再跑整树）


In [ ]:
def bypass_or_branch(parent_soc, children):
    # children: 列表 (label, soc, n_conflicts)
    same = [c for c in children if c[1] == parent_soc]
    if same:
        pick = min(same, key=lambda c: c[2])
        return "bypass", pick
    return "branch", children


print("等 SOC 少冲突", bypass_or_branch(10, [("A", 10, 1), ("B", 12, 0)]))
print("双方都涨", bypass_or_branch(10, [("A", 11, 0), ("B", 12, 0)]))
print("双方都不涨", bypass_or_branch(10, [("A", 10, 2), ("B", 10, 1)]))


In [ ]:
from evals.scenarios import two_agent_swap
from lib.algos.cbs import cbs, cbs_enhanced

g, starts, goals = two_agent_swap()
plain = cbs(g, starts, goals)
enh = cbs_enhanced(g, starts, goals)
print("朴素高层", plain["high"], "树", plain.get("tree"), "SOC", plain["soc"])
print("增强高层", enh["high"], "树", enh.get("tree"), "SOC", enh["soc"])
print("SOC 应相同", plain["soc"] == enh["soc"])


## 7. 逐行（库 `enhanced=True`）

| 行 | 作用 |
|----|------|
| 先生成全部孩子 | 才能比较 SOC |
| `same = [c for c in children if c[2]==node.cost]` | 不涨价的孩子 |
| 只 push `same[0]` | 简化 bypass |
| else 全 push | 近似 cardinal 全拆 |

读 `cbs(..., enhanced=True)` 的 `if same:`。工业代码还有 WDG 启发式（pairwise 依赖图当高层 h）。

## 常见 bug

1. bypass 时没检查冲突数是否真下降，可能循环。
2. 把非基数冲突当基数，乱加约束——本实现不加额外约束，只少压节点，不破坏最优。
3. 以为增强可以次优换速度——那是 ECBS，下一课。

## 条目小结

| 项目 | 内容 |
|------|------|
| 目标 | 少扩展、仍最优 |
| bypass | 等 SOC 孩子替换当前 |
| cardinal | 双方都涨价的冲突优先 |
| 下一课 | 允许 w·OPT 换速度 |

## 练习

1. 为何 bypass 不破坏最优？用下界说话。
2. 两个孩子 SOC 都不涨，选冲突更少的那个。为什么？
3. 走廊对称冲突若一次只禁一格，树会怎样长？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), enh, "CBS+")
studio
